### RAG pipelines - Data Ingestion to vector DB pipeline


In [2]:
!pip install -U langchain-text-splitters


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/mv/nlgffl8x5n93sn5ks85sqb_00000gn/T/ipykernel_29672/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/Users/kiruthikam/Downloads/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)      #add source information to metdata
            print(f" Loaded Successfully {len(documents)} pages")
        except Exception as e:
            print(f" Error:{e}")
    print(f"\n Total documents loaded : {len(all_documents)}")
    return all_documents
all_pdf_documents = process_all_pdfs("../../data")


Found 2 PDF files to process

Processing: GenAi.pdf
 Loaded Successfully 2 pages

Processing: TCS_prep.pdf
 Loaded Successfully 26 pages

 Total documents loaded : 28


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'INT446 - GENERATIVE AI WITH LARGE LANGUAGE MODELS-1', 'source': '../../data/pdf/GenAi.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'GenAi.pdf', 'file_type': 'pdf'}, page_content='M.Sc.  (Data  Science)   \nL  T  P  C   3  1  0  4   \nCourse  Code:  INT446   \nGENERATIVE  AI  WITH  LARGE  LANGUAGE  MODELS   Course  objectives:   The  course  helps  the  learners  to  Analyze  the  basic  concepts  of  Large  Language  Models  and  \nits\n \napplications\n \nin\n \nNatural\n \nLanguage\n \nProcessing\n \nand\n \napply\n \nthe\n \nLarge\n \nLanguage\n \nModels\n \nin\n \nvarious\n \nlanguage\n-related\n \napplications\n \nsuch\n \nas\n \ntext\n \nclassification,\n \nlanguage\n \ntranslation,\n \ncontent\n \ngeneration,\n \nChatBot,\n \netc.\n \n \nUNIT  -  I  15  Periods  Generative  AI  Use  Cases,  Fundamentals,  and  Project  Life  Cycle:  \nUs

In [8]:
## Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

# Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [10]:
chunks=split_documents(all_pdf_documents)


Split 28 documents into 55 chunks

Example chunk:
Content: M.Sc.  (Data  Science)   
L  T  P  C   3  1  0  4   
Course  Code:  INT446   
GENERATIVE  AI  WITH  LARGE  LANGUAGE  MODELS   Course  objectives:   The  course  helps  the  learners  to  Analyze  the ...
Metadata: {'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'INT446 - GENERATIVE AI WITH LARGE LANGUAGE MODELS-1', 'source': '../../data/pdf/GenAi.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'GenAi.pdf', 'file_type': 'pdf'}


In [11]:
chunks

[Document(metadata={'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'INT446 - GENERATIVE AI WITH LARGE LANGUAGE MODELS-1', 'source': '../../data/pdf/GenAi.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'GenAi.pdf', 'file_type': 'pdf'}, page_content='M.Sc.  (Data  Science)   \nL  T  P  C   3  1  0  4   \nCourse  Code:  INT446   \nGENERATIVE  AI  WITH  LARGE  LANGUAGE  MODELS   Course  objectives:   The  course  helps  the  learners  to  Analyze  the  basic  concepts  of  Large  Language  Models  and  \nits\n \napplications\n \nin\n \nNatural\n \nLanguage\n \nProcessing\n \nand\n \napply\n \nthe\n \nLarge\n \nLanguage\n \nModels\n \nin\n \nvarious\n \nlanguage\n-related\n \napplications\n \nsuch\n \nas\n \ntext\n \nclassification,\n \nlanguage\n \ntranslation,\n \ncontent\n \ngeneration,\n \nChatBot,\n \netc.\n \n \nUNIT  -  I  15  Periods  Generative  AI  Use  Cases,  Fundamentals,  and  Project  Life  Cycle:  \nUs

### Embedding and VectorStoreDB

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
          if not self.model:
            raise ValueError("Model not loaded")
          print(f"Generating embeddings for {len(texts)} texts...")
          embeddings = self.model.encode(texts, show_progress_bar=True)
          print(f"Generated embeddings with shape: {embeddings.shape}")
          return embeddings
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8487.16it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [14]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(
                f"Vector store initialized. Collection: "
                f"{self.collection_name}"
            )
            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents to vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to vector store"
            )
            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise
vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [15]:
chunks

[Document(metadata={'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'INT446 - GENERATIVE AI WITH LARGE LANGUAGE MODELS-1', 'source': '../../data/pdf/GenAi.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'GenAi.pdf', 'file_type': 'pdf'}, page_content='M.Sc.  (Data  Science)   \nL  T  P  C   3  1  0  4   \nCourse  Code:  INT446   \nGENERATIVE  AI  WITH  LARGE  LANGUAGE  MODELS   Course  objectives:   The  course  helps  the  learners  to  Analyze  the  basic  concepts  of  Large  Language  Models  and  \nits\n \napplications\n \nin\n \nNatural\n \nLanguage\n \nProcessing\n \nand\n \napply\n \nthe\n \nLarge\n \nLanguage\n \nModels\n \nin\n \nvarious\n \nlanguage\n-related\n \napplications\n \nsuch\n \nas\n \ntext\n \nclassification,\n \nlanguage\n \ntranslation,\n \ncontent\n \ngeneration,\n \nChatBot,\n \netc.\n \n \nUNIT  -  I  15  Periods  Generative  AI  Use  Cases,  Fundamentals,  and  Project  Life  Cycle:  \nUs

In [16]:
## Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

embeddings=embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 55 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.36it/s]

Generated embeddings with shape: (55, 384)
Adding 55 documents to vector store...
Successfully added 55 documents to vector store
Total documents in collection: 55


### Retriver pipeline from vectorstore 

In [17]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(
            f"Top K: {top_k}, Score threshold: {score_threshold}"
        )

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score
                    # ChromaDB uses cosine distance
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} documents "
                    f"(after filtering)"
                )

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriver=RAGRetriever(vectorstore,embedding_manager)
rag_retriver

In [18]:
rag_retriver.retrieve("what is TCs prep is all you need")

Retrieving documents for query: 'what is TCs prep is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.25it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_292b7d44_6',
  'content': 'TCS Placement Preparation\n Handbook\n Frequently Asked Questions Edition — NQT | Digital | Prime | Atlas\n Hiring\n A Frequently Asked Questions companion covering Quantitative\nAptitude, Logical Reasoning, Verbal Ability, Coding, Data Structures &\nAlgorithms, DBMS, Operating Systems, Computer Networks, OOP, HR\n Interview and a 30-Day Study Plan.\nPrepared for: Kiki\nM.Sc. Data Science, SASTRA Deemed University\nTarget: TCS NQT / TCS Digital / TCS Prime / TCS Atlas Hiring',
  'metadata': {'total_pages': 26,
   'file_type': 'pdf',
   'creator': '(unspecified)',
   'source_file': 'TCS_prep.pdf',
   'keywords': '',
   'subject': '(unspecified)',
   'doc_index': 6,
   'trapped': '/False',
   'content_length': 469,
   'creationdate': '2026-07-03T11:51:05+00:00',
   'page': 0,
   'moddate': '2026-07-03T11:51:05+00:00',
   'page_label': '1',
   'author': 'TCS Placement Trainer',
   'title': 'TCS Placement Preparation Handbook',
   'source': '../../da

In [19]:
rag_retriver.retrieve("tell me about concepts to cover for TCS preparation")


Retrieving documents for query: 'tell me about concepts to cover for TCS preparation'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.55it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_9ceca9cb_7',
  'content': 'TCS Placement Preparation Handbook\nPage 1\nPrepared for Kiki\nTable of Contents\n1. Frequently Asked Questions — Quantitative Aptitude\n2. Frequently Asked Questions — Logical Reasoning\n3. Frequently Asked Questions — Verbal Ability\n4. Frequently Asked Questions — Coding Fundamentals (C / C++ / Java / Python)\n5. Frequently Asked Questions — Data Structures\n6. Frequently Asked Questions — Algorithms\n7. Frequently Asked Coding Practice Questions (with Solutions)\n8. Frequently Asked Questions — DBMS / Operating System / Computer Networks\n9. Frequently Asked Questions — OOP Concepts (Python & Java)\n10. Frequently Asked HR Interview Questions\n11. 30-Day Study Plan',
  'metadata': {'source': '../../data/pdf/TCS_prep.pdf',
   'total_pages': 26,
   'subject': '(unspecified)',
   'page': 1,
   'title': 'TCS Placement Preparation Handbook',
   'author': 'TCS Placement Trainer',
   'creator': '(unspecified)',
   'source_file': 'TCS_prep.pdf',
   '

### Integration VectorDB contect pipeline with LLM Output

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found. Please set GROQ_API_KEY in your .env file.")
groq_model = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

# 1. Initialize Groq LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model=groq_model,
    temperature=0.1,
    max_tokens=1024
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # Generate answer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)
    return response.content

print("ChatGroq LLM initialized with model:", groq_model)


### 1. Test Simple RAG with TCS Preparation Query


In [ ]:
response = rag_simple("tell me about concepts to cover for TCS preparation", rag_retriver, llm, top_k=3)
print("RAG Response:\n", response)


### 2. Test RAG with DBMS ACID Properties Query


In [ ]:
response = rag_simple("What are the ACID properties in DBMS?", rag_retriver, llm, top_k=3)
print("RAG Response:\n", response)


### 3. Test RAG with Generative AI Course Units Query


In [ ]:
response = rag_simple("What units are covered in the Generative AI course?", rag_retriver, llm, top_k=3)
print("RAG Response:\n", response)


### 4. Advanced: LangChain LCEL RAG Chain with Prompt Template & Sources


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define structured ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI tutor. Answer questions accurately based ONLY on the given context. Cite sources when possible.\n\nContext:\n{context}"),
    ("human", "{question}")
])

# Build LCEL chain
rag_chain = prompt_template | llm | StrOutputParser()

# Run query
query = "Explain the difference between Stack and Queue according to the document."
docs = rag_retriver.retrieve(query, top_k=3)
context = rag_retriver.format_context(docs) if hasattr(rag_retriver, "format_context") else "\n\n".join([d["content"] for d in docs])

answer = rag_chain.invoke({"context": context, "question": query})
print("Answer:", answer)


### 5. Modular RAG Pipeline from `src.rag`
You can also run the entire end-to-end pipeline using the modular library created in `src/rag`:


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent / "src"))

from rag import RAGPipeline

# Initialize modular pipeline
pipeline = RAGPipeline()

# Query directly
result = pipeline.query("What are the main advantages of Python?")
print("Answer:", result["answer"])
print("\nSources referenced:", [s["source_file"] for s in result["sources"]])
